# GEAP Platform Tour — Simple Agent, Agent Identity, Gateway, User-Scoped Memory & Sessions

A compact, **run-to-completion** walkthrough of the core Gemini Enterprise Agent Platform (GEAP)
building blocks, using one **simple** ADK agent on **Gemini 3.7 Flash**:

1. **Build** a minimal agent (one tool + memory) — `simple_agent/agent.py`.
2. **Deploy** it to **Agent Engine** with **Agent Identity (SPIFFE)**.
3. **Configure** an **Agent Gateway** attachment (ingress + egress) and explain how it governs traffic.
4. **Session service** — hold multi-turn conversations, per user.
5. **Memory Bank scoped at the user level** — the agent remembers a user across sessions, and one
   user cannot see another user's memories.

> **Model:** Gemini 3.x is served only on the **global** endpoint, so the agent uses the *native* ADK
> `Gemini` model pinned to `location="global"` (not LiteLLM, which garbles Gemini-3 thought signatures).
>
> **Scope note:** the agent is deployed with `AGENT_IDENTITY` (works today). The gateway *attachment*
> is shown as configuration + explained but **not executed** — attaching `agent_gateway_config`
> currently requires an AI-Platform Private-Preview enrollment this project doesn't have, and an
> attached egress gateway breaks the runtime's own session/memory calls. This mirrors
> `gateway_sdk_demo.ipynb`.

## Setup

In [1]:
import os, json, time, asyncio, warnings, logging

# Run from the repo root so `simple_agent` imports and `.env` resolve.
for _ in range(6):
    if os.path.exists("simple_agent/agent.py"):
        break
    os.chdir("..")

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

warnings.filterwarnings("ignore")
for _n in ("google_adk", "google.adk", "google.auth", "grpc", "httpx", "urllib3", "google.genai"):
    logging.getLogger(_n).setLevel(logging.CRITICAL)

import vertexai
from vertexai import agent_engines
from vertexai._genai import types as ae_types

# Prefer GCP_PROJECT_ID (from .env); the ambient GOOGLE_CLOUD_PROJECT may point elsewhere.
PROJECT  = os.environ.get("GCP_PROJECT_ID", "geap-smoke-test")
LOCATION = os.environ.get("GCP_REGION", "us-central1")            # Agent Engine + sessions are REGIONAL
BUCKET   = os.environ.get("GCP_STAGING_BUCKET", f"{PROJECT}-geap-staging")
MODEL    = os.environ.get("AGENT_MODEL", "gemini-3.7-flash")
GATEWAY        = f"projects/{PROJECT}/locations/{LOCATION}/agentGateways/geap-workshop-gateway"
GATEWAY_EGRESS = f"projects/{PROJECT}/locations/{LOCATION}/agentGateways/geap-workshop-gateway-egress"

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "1")
os.environ["GCP_PROJECT_ID"] = PROJECT
os.environ["AGENT_MODEL"] = MODEL

vertexai.init(project=PROJECT, location=LOCATION)                 # required before agent_engines.get()
client = vertexai.Client(project=PROJECT, location=LOCATION, http_options=dict(api_version="v1beta1"))
print(f"project={PROJECT}  region={LOCATION}  model={MODEL}")
print("bucket:", BUCKET)

project=geap-smoke-test  region=us-central1  model=gemini-3.7-flash
bucket: geap-smoke-test-geap-staging


## 1. Build a simple agent (Gemini 3.7 Flash)

The agent lives in `simple_agent/agent.py` — a self-contained module (no MCP, no `src` deps) so it
deploys cleanly. It has one plain-Python tool (`get_current_time`), a `PreloadMemoryTool` (recalls the
current user's memories each turn), and an `after_agent_callback` that persists the session to Memory
Bank. The model is resolved through the **native `Gemini`** path pinned to the **global** endpoint.

In [2]:
from simple_agent.agent import root_agent

m = root_agent.model
print("agent name      :", root_agent.name)
print("model class     :", type(m).__name__)                 # Gemini (native), NOT LiteLlm
print("model id        :", getattr(m, "model", m))
print("client_kwargs   :", getattr(m, "client_kwargs", None))  # -> location=global (Gemini 3.x)
print("tools           :", [getattr(t, "name", type(t).__name__) for t in root_agent.tools])
print("after_callback  :", getattr(root_agent.after_agent_callback, "__name__", None))

agent name      : simple_time_agent
model class     : Gemini
model id        : gemini-3.7-flash
client_kwargs   : {'vertexai': True, 'location': 'global', 'project': 'geap-smoke-test'}
tools           : ['function', 'preload_memory']
after_callback  : save_memories_callback


### Why the native Gemini + global path?
Gemini 3.x models (like `gemini-3.7-flash`) are **only** served on Vertex's `global` endpoint, so a
plain regional call 404s. We use ADK's native `Gemini(model=..., client_kwargs={"vertexai":True,
"location":"global","project":...})` — a **per-model** pin, so the Agent Engine itself still lives in
`us-central1` while model calls go global. We deliberately avoid LiteLLM here: its Vertex adapter
turns Gemini-3 *thought signatures* into malformed function calls, and a tool-using agent then never
returns a real answer.

## 2. Deploy to Agent Engine with Agent Identity (SPIFFE)

`vertexai.Client().agent_engines.create(agent=..., config={...})` cloud-pickles the agent, ships
`simple_agent/` via `extra_packages`, and provisions a managed **Agent Engine** (reasoning engine)
that auto-wires a **session service** and a **Memory Bank** at its own id. We set
`identity_type=AGENT_IDENTITY` to give the engine a keyless **SPIFFE** workload identity.

This cell is **idempotent** — it reuses an existing `simple_time_agent` engine if one exists, else
creates one (first create takes a few minutes).

In [3]:
DISPLAY = "simple_time_agent"

def find_engine(display_name):
    for e in agent_engines.list():
        dn = getattr(e, "display_name", None) or getattr(getattr(e, "api_resource", None), "display_name", None)
        if dn == display_name:
            return getattr(e, "resource_name", None) or e.api_resource.name
    return None

engine_resource = find_engine(DISPLAY)
if engine_resource:
    print("Reusing existing engine:", engine_resource)
else:
    config = {
        "staging_bucket": f"gs://{BUCKET}",
        "display_name": DISPLAY,
        "extra_packages": ["simple_agent"],          # resolved against the CWD (repo root)
        "requirements": [
            "google-cloud-aiplatform[adk,agent-engines]>=1.162.0",
            "google-adk==2.5.0",                     # EXACT pin: must match the pickling ADK
            "google-genai>=2.14.0",
            "google-auth>=2.52.0",
            "cloudpickle>=3.0,<4.0",
            "pydantic>=2.12.5",
        ],
        # NOTE: GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION are RESERVED by the runtime — do not set them.
        "env_vars": {
            "GOOGLE_GENAI_USE_VERTEXAI": "1",
            "GCP_PROJECT_ID": PROJECT,
            "AGENT_MODEL": MODEL,
            "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
        },
        "labels": {"app": "geap-workshop", "component": "tutorial-agent"},
        "identity_type": ae_types.IdentityType.AGENT_IDENTITY,   # SPIFFE workload identity
    }
    remote = client.agent_engines.create(agent=root_agent, config=config)
    engine_resource = getattr(remote, "resource_name", None) or remote.api_resource.name
    print("Created engine:", engine_resource)

ENGINE_ID = engine_resource.split("/")[-1]
print("ENGINE_ID:", ENGINE_ID)

Reusing existing engine: projects/907173573292/locations/us-central1/reasoningEngines/6407454038643703808
ENGINE_ID: 6407454038643703808


### Agent Identity (SPIFFE) — how it works

With `identity_type=AGENT_IDENTITY`, the deployed engine runs under a **keyless, SPIFFE-based workload
identity** instead of a long-lived service-account key. The identity is *resource-scoped* to this exact
engine, so every action it takes (model calls, session/memory writes) is auditable to a specific agent
principal of the form:

```
agents.global.<ORG>.system.id.goog/resources/aiplatform/projects/<PROJECT>/locations/<LOCATION>/reasoningEngines/<ENGINE_ID>
```

Under the hood a global **Workload Identity Pool** (`agent-pool`) + an OIDC provider (issuer
`https://agents.global.org-<ORG>.system.id.goog`) trust that principal, and a **principalSet** IAM
binding grants it `roles/aiplatform.user` — which is what lets the runtime call Vertex APIs (including
its own session and memory writes) **without any service-account key**. Let's read the engine's
`effectiveIdentity` live:

In [4]:
eng = client.agent_engines.get(name=engine_resource)
spec = eng.api_resource.spec
print("identity_type      :", getattr(spec, "identity_type", None))
print("effective_identity :", getattr(spec, "effective_identity", None))

identity_type      : IdentityType.AGENT_IDENTITY
effective_identity : agents.global.org-1060412978793.system.id.goog/resources/aiplatform/projects/907173573292/locations/us-central1/reasoningEngines/6407454038643703808


## 3. Configure an Agent Gateway attachment (ingress + egress)

The **Agent Gateway** is the network boundary that governs traffic to/from an agent:
- **Ingress** (`client_to_agent`) — governs *clients → agent* calls (IAM/CEL, IAP, Model Armor, Semantic Governance).
- **Egress** (`agent_to_anywhere`) — governs the *agent → external* calls.

A gateway is bound at **deploy time** via `agent_gateway_config` (it can't be PATCHed onto a running
engine). Here we build and show the config; see the note below on why we don't execute the attach.

In [5]:
agent_gateway_config = {
    "client_to_agent_config":   {"agent_gateway": GATEWAY},          # ingress
    "agent_to_anywhere_config": {"agent_gateway": GATEWAY_EGRESS},   # egress
}
print(json.dumps(agent_gateway_config, indent=2))
print()
print("To attach at deploy time you would add `agent_gateway_config` (+ identity) to the create/update")
print("config, e.g.:")
print("  ENABLE_AGENT_GATEWAY=1 ENABLE_AGENT_IDENTITY=1 \\")
print("    uv run python -m src.deploy.deploy_agents <agent> --update")
print("\nNote: `--update` performs a REDEPLOY that binds the gateway at create/update time.")
print("It is not a live/in-place patch — a gateway cannot be attached to an already-running engine.")

{
  "client_to_agent_config": {
    "agent_gateway": "projects/geap-smoke-test/locations/us-central1/agentGateways/geap-workshop-gateway"
  },
  "agent_to_anywhere_config": {
    "agent_gateway": "projects/geap-smoke-test/locations/us-central1/agentGateways/geap-workshop-gateway-egress"
  }
}

To attach at deploy time you would add `agent_gateway_config` (+ identity) to the create/update
config, e.g.:
  ENABLE_AGENT_GATEWAY=1 ENABLE_AGENT_IDENTITY=1 \
    uv run python -m src.deploy.deploy_agents <agent> --update

Note: `--update` performs a REDEPLOY that binds the gateway at create/update time.
It is not a live/in-place patch — a gateway cannot be attached to an already-running engine.


### Why config-only here
Attaching `agent_gateway_config` currently requires an **AI-Platform Private-Preview** enrollment
(separate from the Network Services one). Without it, `create`/`update` starts the deployment then
fails with `code: 13 INTERNAL`. Additionally, an attached **egress** gateway intercepts the runtime's
own outbound call to `aiplatform.googleapis.com`, which breaks session/memory creation. So — to keep
this tutorial runnable end-to-end while still demonstrating the governance model — we build and explain
the config but deploy the agent **without** the gateway attached (identity is on). The three policy
layers (IAM/CEL, Semantic Governance, IAP + Model Armor) are covered in `gateway_sdk_demo.ipynb`.

## 4. Session service — multi-turn conversations

The deployed Agent Engine exposes a **session service**: `create_session(user_id=...)` returns a
session whose `id` you pass back to `stream_query(..., session_id=...)`. Reusing the same `session_id`
across turns makes the runtime rehydrate the full event history, so the conversation is **stateful**.
We use the `vertexai.agent_engines.get(...)` proxy, which binds the deployed engine's operations.

In [6]:
remote_app = agent_engines.get(engine_resource)
USER_A = "alice"

def say(text, user, session_id):
    """Send one turn; collect the agent's final text."""
    final = ""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        for ev in remote_app.stream_query(message=text, user_id=user, session_id=session_id):
            for part in (ev.get("content") or {}).get("parts", []):
                if part.get("text"):
                    final += part["text"]
    print(f"[{user}] {text}\n   -> {final.strip()[:300]}\n")
    return final

# One session, three turns that build on each other (proves per-session state persists).
session_a = remote_app.create_session(user_id=USER_A)
sid_a = session_a["id"]                                   # the session id key is "id"
print("created session:", sid_a, "\n")

say("Hi! My preferred name is Sam and I love window seats. What time is it in UTC right now?", USER_A, sid_a)
say("What's my preferred name?", USER_A, sid_a)           # in-session recall
say("And which seat do I prefer?", USER_A, sid_a)

# List this user's sessions, then read the persisted event history for the session.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    listed = remote_app.list_sessions(user_id=USER_A)
    ids = [s.get("id") for s in (listed.get("sessions") or [])]
    print("alice sessions:", ids)
    full = remote_app.get_session(user_id=USER_A, session_id=sid_a)
print("\nevent history for", sid_a, ":")
for e in (full.get("events") or []):
    txt = "".join((p.get("text") or "") for p in (e.get("content") or {}).get("parts", []))
    if txt.strip():
        print(f"  [{e.get('author')}] {txt.strip()[:120]}")

created session: 4125856841370435584 



[alice] Hi! My preferred name is Sam and I love window seats. What time is it in UTC right now?
   -> Hi Sam! I've made a note that you prefer window seats. 

The current time in UTC is 2:56 PM (14:56) on August 19, 2026.



[alice] What's my preferred name?
   -> Your preferred name is Sam!



[alice] And which seat do I prefer?
   -> You prefer window seats!



alice sessions: ['4125856841370435584', '8221036277533769728', '5844261574188990464', '2167072478440980480', '7713255419547746304', '293575033454854144', '3468612770751053824', '1516865282239365120']



event history for 4125856841370435584 :
  [user] Hi! My preferred name is Sam and I love window seats. What time is it in UTC right now?
  [simple_time_agent] Hi Sam! I've made a note that you prefer window seats. 

The current time in UTC is 2:56 PM (14:56) on August 19, 2026.
  [user] What's my preferred name?
  [simple_time_agent] Your preferred name is Sam!
  [user] And which seat do I prefer?
  [simple_time_agent] You prefer window seats!


## 5. Memory Bank scoped at the user level

Every Agent Engine has a built-in **Memory Bank**. Memories are partitioned by a **scope** of
`{app_name, user_id}` — i.e. **per user** (for a deployed engine `app_name` is the engine's own id, so
the effective isolation key is the `user_id`). Our agent's `after_agent_callback`
(`add_session_to_memory`) persists sessions to this Memory Bank automatically, but that extraction is
**eventually consistent**. So for a deterministic demo we write a durable memory for Alice **explicitly**
via the same Memory Bank API (`memories.create(..., config={"wait_for_completion": True})`), then prove
user-scoping: **Alice recalls, Bob sees nothing.**

In [7]:
ENGINE_APP = ENGINE_ID                 # a deployed engine's memory app_name == its own id
FACT = "The user's preferred name is Sam and they prefer window seats."
QUERY = "preferred name and seating preference"

def mem_texts(resp):
    """Pull memory text out of an async_search_memory response (dict shape)."""
    mems = resp.get("memories") if isinstance(resp, dict) else (getattr(resp, "memories", None) or [])
    out = []
    for m in (mems or []):
        content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
        parts = (content or {}).get("parts", []) if isinstance(content, dict) else (getattr(content, "parts", None) or [])
        txt = " ".join((p.get("text") if isinstance(p, dict) else getattr(p, "text", None)) or "" for p in parts).strip()
        out.append(txt or str(m))
    return out

async def search(user, query):
    return await remote_app.async_search_memory(user_id=user, query=query)

# Idempotent write: memories.create does NOT dedup, so only write if Alice doesn't already have it
# (keeps re-runs of this notebook from accumulating identical facts).
existing = mem_texts(await search(USER_A, QUERY))
if not any("window seat" in t.lower() for t in existing):
    op = client.agent_engines.memories.create(
        name=engine_resource,
        fact=FACT,
        scope={"app_name": ENGINE_APP, "user_id": USER_A},   # USER-scoped
        config={"wait_for_completion": True},                # synchronous
    )
    print("memory write done:", getattr(op, "done", op))
else:
    print("memory already present for", USER_A, "— skipping write")

alice_mem = list(dict.fromkeys(mem_texts(await search(USER_A, QUERY))))   # dedup for display
print(f"\nAlice's memories ({len(alice_mem)}):")
for t in alice_mem:
    print("   +", t[:160])

# Prove user-scoping: Bob queries the SAME engine and sees nothing of Alice's.
bob_mem = mem_texts(await search("bob", QUERY))
print(f"\nBob's memories ({len(bob_mem)}):", bob_mem or "(none — memory is scoped per user)")

memory already present for alice — skipping write



Alice's memories (1):
   + The user's preferred name is Sam and they prefer window seats.



Bob's memories (0): (none — memory is scoped per user)


In [8]:
# 3) Cross-session recall: a BRAND-NEW Alice session; the agent recalls her prefs via PreloadMemoryTool.
session_a2 = remote_app.create_session(user_id=USER_A)
print("new alice session:", session_a2["id"], "\n")
say("Remind me — what name do I go by, and what seat do I prefer?", USER_A, session_a2["id"])

new alice session: 5976836288219709440 



[alice] Remind me — what name do I go by, and what seat do I prefer?
   -> Your name is Sam, and you prefer window seats!



'Your name is Sam, and you prefer window seats!'

### How user-scoped memory works
- The runtime auto-wires `VertexAiMemoryBankService` at the engine's own id (no config needed) —
  `add_session_to_memory()` writes under scope `{app_name=<engine>, user_id=<session user>}`, and
  `PreloadMemoryTool` searches with that same scope every turn. Because scope is an **exact match on
  `user_id`**, a different user recalls nothing — that *is* the isolation boundary.
- **Session vs memory:** the **session service** holds one conversation's turns (short-term, per
  `session_id`); the **Memory Bank** holds durable facts about a **user** across all their sessions.
  Together they give an agent both working memory and long-term, user-scoped memory.

## Recap

You built one simple **Gemini 3.7** agent and used it to exercise the GEAP platform end-to-end:

| Capability | What you did |
|---|---|
| **Simple agent** | `simple_agent/agent.py` — native Gemini (global), one tool + memory |
| **Agent Engine deploy** | `client.agent_engines.create(...)`, idempotent |
| **Agent Identity (SPIFFE)** | `identity_type=AGENT_IDENTITY`; read the resource-scoped `effectiveIdentity` |
| **Agent Gateway** | built + explained the ingress/egress `agent_gateway_config` (attach is Private-Preview gated) |
| **Session service** | `create_session` + multi-turn `stream_query(session_id=...)`, `list_sessions`, `get_session` |
| **User-scoped memory** | Memory Bank `{app_name, user_id}` scope — Alice recalls across sessions; Bob sees nothing |

Next: govern this agent's traffic in `gateway_sdk_demo.ipynb`, or register it for discovery in
`registry_sdk_demo.ipynb`.